# 04 · Layer 2：Runner、Session 與 State

Layer 1 講的是「一個 agent 由什麼組成」。Layer 2 講的是「它怎麼跑起來」。

```
                   ┌──────── Runner ────────┐
  你的訊息 ──────▶ │                        │ ──▶ 事件串流
                   │  agent                 │
                   │  session_service   ← 對話狀態
                   │  memory_service    ← 長期記憶（第 05 章）
                   │  artifact_service  ← 檔案（第 05 章）
                   │  plugins           ← 橫切關注（第 06 章）
                   └────────────────────────┘
```

本章專注在 **Session 與 State**：agent 之所以「記得」上一句話，全靠它們。

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. Session 是什麼

一個 `Session` 就是**一段對話**，身上有四樣東西：

| 屬性 | 內容 |
|---|---|
| `id` | 這段對話的識別碼 |
| `app_name` / `user_id` | 這段對話屬於哪個應用、哪個使用者 |
| `events` | 完整的事件歷史（每一句話、每一次工具呼叫） |
| `state` | 一個 dict，放結構化資料 |

`events` 是「發生過什麼」，`state` 是「現在的狀況」。兩者用途完全不同。

In [2]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

APP = "concept_track"
USER = "student"

session_service = InMemorySessionService()

assistant = LlmAgent(
    name="assistant",
    model=get_model(),
    instruction="你是助理，用繁體中文簡短回答。",
)

runner = Runner(agent=assistant, app_name=APP, session_service=session_service)

session = await session_service.create_session(app_name=APP, user_id=USER)
print(f"id       : {session.id}")
print(f"app/user : {session.app_name} / {session.user_id}")
print(f"events   : {len(session.events)} 筆")
print(f"state    : {dict(session.state)}")

id       : c67b17f9-c13a-4a87-94d4-2e0926ffbd3f
app/user : concept_track / student
events   : 0 筆
state    : {}


In [3]:
await ask(runner, "我最喜歡的程式語言是 Rust。", session_id=session.id)
await ask(runner, "我剛剛說我喜歡什麼？", session_id=session.id)

session = await session_service.get_session(app_name=APP, user_id=USER, session_id=session.id)
print(f"對話兩輪之後，events = {len(session.events)} 筆")
for ev in session.events:
    kind = "user" if ev.author == "user" else ev.author
    text = ""
    if ev.content and ev.content.parts:
        text = "".join(p.text or "" for p in ev.content.parts)[:40]
    print(f"  [{kind:10s}] {text}")

對話兩輪之後，events = 4 筆
  [user      ] 我最喜歡的程式語言是 Rust。
  [assistant ] 很高興聽到！Rust 是一門兼具高效能與記憶體安全的優秀語言，特別適合系統程式設
  [user      ] 我剛剛說我喜歡什麼？
  [assistant ] 你說你最喜歡的程式語言是 Rust。


**agent 的「記憶」其實就是 `events` 被整包重送給模型。** 沒有魔法。
這也直接說明了為什麼長對話會越來越貴——每一輪都在重送全部歷史。
解法（壓縮、快取）在第 06 章與 30 天實作的 Day 10 / 11。

## 2. State：agent 之間傳資料的管道

`state` 是一個 dict，但 key 的**前綴**決定它活多久、給誰看：

| 前綴 | 作用域 | 什麼時候用 |
|---|---|---|
| （無） | 這個 session | 這段對話的暫存資料 |
| `user:` | 這個使用者的所有 session | 使用者偏好、個人設定 |
| `app:` | 整個應用的所有使用者 | 全域設定、公告 |
| `temp:` | **只活在這一次呼叫，不會被寫進 session** | 中間計算結果、敏感資料 |

In [4]:
from google.adk.tools import ToolContext


def remember_preference(key: str, value: str, tool_context: ToolContext) -> dict:
    """記住使用者的一項偏好設定。

    Args:
        key: 偏好項目名稱，例如 'language'、'tone'。
        value: 偏好的值。
    """
    tool_context.state[f"user:{key}"] = value      # 跨 session 保留
    tool_context.state["last_updated_key"] = key   # 只在這個 session
    tool_context.state["temp:debug_note"] = "這行不會被存下來"
    return {"ok": True, "key": key, "value": value}


pref_agent = LlmAgent(
    name="pref_agent",
    model=get_model(),
    instruction="使用者說出偏好時，呼叫 remember_preference 記下來，然後用一句話確認。",
    tools=[remember_preference],
)

pref_runner = Runner(agent=pref_agent, app_name=APP, session_service=session_service)
sid_a = await new_session(pref_runner)

print(await ask(pref_runner, "我希望你以後都用比較正式的語氣回答", session_id=sid_a))
print("\n--- session A 的 state ---")
print_state(await peek_state(pref_runner, sid_a))

本系統已記錄您的偏好，未來將以較為正式之語氣為您解答。

--- session A 的 state ---
  last_updated_key: tone
  user:tone: formal


注意兩件事：

1. `temp:debug_note` **不見了**——`temp:` 前綴的東西不會被寫進 session。
2. `user:tone` 和 `last_updated_key` 都在。

### `user:` 真的跨 session 嗎？開第二段對話驗證

In [5]:
sid_b = await new_session(pref_runner)
print("--- 全新的 session B ---")
print_state(await peek_state(pref_runner, sid_b))

--- 全新的 session B ---
  user:tone: formal


`user:tone` 出現在一個全新的 session 裡，`last_updated_key` 沒有。
這就是作用域的差別，而且它是 ADK 內建的，你不用自己寫同步邏輯。

## 3. `output_key`：把 agent 的回覆自動存進 state

這是多 agent pipeline 傳資料的**標準做法**，第 07、08 章會大量用到。

`output_key="x"` 的意思是：這個 agent 的最終回覆，自動存到 `state["x"]`。
下一個 agent 就能在 instruction 裡用 `{x?}` 讀到。

In [6]:
summarizer = LlmAgent(
    name="summarizer",
    model=get_model(),
    instruction="把使用者給的文字濃縮成一句話，只回那句話。",
    output_key="summary",
)

translator = LlmAgent(
    name="translator",
    model=get_model(),
    instruction="把下面這句中文翻成英文，只回英文：\n\n{summary?}",
)

pipeline_runner = Runner(agent=summarizer, app_name=APP, session_service=session_service)
sid_c = await new_session(pipeline_runner)

long_text = (
    "Google ADK 是一套開發 AI Agent 的框架。它把 agent、工具、執行環境、"
    "多 agent 協作都模組化，讓開發者不用自己從零打造 agent loop，"
    "也提供評估、部署、觀測等上線需要的工具。"
)
print("摘要:", await ask(pipeline_runner, long_text, session_id=sid_c))
print("\nstate:")
print_state(await peek_state(pipeline_runner, sid_c))

摘要: Google ADK 是一套模組化且具備完整上線工具的 AI Agent 開發框架。

state:
  summary: Google ADK 是一套模組化且具備完整上線工具的 AI Agent 開發框架。
  user:tone: formal


In [7]:
# 換 agent，同一個 session —— translator 讀得到 {summary}
pipeline_runner.agent = translator
print("翻譯:", await ask(pipeline_runner, "翻譯", session_id=sid_c))

翻譯: Google ADK is a modular AI Agent development framework equipped with comprehensive production-ready tools.


這裡我們手動換了 `runner.agent`，只是為了讓你看清楚資料是怎麼流的。
實務上會用 `SequentialAgent` 自動串（第 07 章）。

## 4. 不要直接改 `session.state`

這是很常見的錯誤。直接改抓回來的 session 物件，**改動不會被記錄下來**：

In [8]:
s = await session_service.get_session(app_name=APP, user_id=USER, session_id=sid_c)
s.state["hacked"] = "我直接改的"

again = await session_service.get_session(app_name=APP, user_id=USER, session_id=sid_c)
print("直接改 session.state 之後再讀一次：", "hacked" in again.state)

直接改 session.state 之後再讀一次： False


在 `InMemorySessionService` 上可能剛好「有效」（因為拿到的是同一個物件），
但換成 `DatabaseSessionService` 就完全不會寫進資料庫。

**正確的改法只有兩種**：

1. 在工具裡用 `tool_context.state[...] = ...`（第 2 節示範過）
2. 用 `EventActions(state_delta=...)` 附在事件上送出

In [9]:
from google.adk.events import Event, EventActions

await session_service.append_event(
    session=await session_service.get_session(app_name=APP, user_id=USER, session_id=sid_c),
    event=Event(
        author="system",
        actions=EventActions(state_delta={"reviewed_by": "sean", "user:plan": "free"}),
    ),
)

print("用 state_delta 寫入之後：")
print_state(await peek_state(pipeline_runner, sid_c))

用 state_delta 寫入之後：
  summary: Google ADK 是一套模組化且具備完整上線工具的 AI Agent 開發框架。
  reviewed_by: sean
  user:tone: formal
  user:plan: free


`state_delta` 的好處是**它本身也是一個事件**——誰在什麼時候改了什麼，
都留在 `events` 裡查得到。這是 ADK 把 state 變更當成事件的核心設計。

## 5. 換掉 InMemory：讓對話活過重開機

`InMemorySessionService` 一重啟 kernel 就全沒了。要持久化，換一個實作就好——
**agent 的程式碼完全不用改**。

| 實作 | 存在哪 | 適用 |
|---|---|---|
| `InMemorySessionService` | 記憶體 | 開發、測試 |
| `SqliteSessionService` | 本機 .db 檔 | 單機小專案 |
| `DatabaseSessionService` | 任何 SQLAlchemy 支援的 DB | 正式環境 |
| `VertexAiSessionService` | Google Cloud | 上雲 |

> **兩個踩坑提醒**：
>
> 1. `SqliteSessionService` 沒有被放進 `google.adk.sessions.__all__`，
>    `from google.adk.sessions import SqliteSessionService` 會 ImportError，
>    要從子模組 `google.adk.sessions.sqlite_session_service` 直接 import。
> 2. 它的參數是 **`db_path`（純路徑）**，不是 `DatabaseSessionService`
>    那種 `db_url="sqlite:///..."`。兩個很像的類別、兩種不同的參數。

In [10]:
from google.adk.sessions import __all__ as sessions_exports

print("google.adk.sessions 匯出的名稱:", sessions_exports)

try:
    from google.adk.sessions import SqliteSessionService  # noqa: F401
except ImportError as exc:
    print(f"\n❌ 直接 import 會失敗: {exc}")

from google.adk.sessions.sqlite_session_service import SqliteSessionService

print("✅ 從子模組 import 成功:", SqliteSessionService.__name__)

google.adk.sessions 匯出的名稱: ['BaseSessionService', 'DatabaseSessionService', 'InMemorySessionService', 'Session', 'State', 'StateSchemaError', 'VertexAiSessionService']

❌ 直接 import 會失敗: cannot import name 'SqliteSessionService' from 'google.adk.sessions' (/Users/linshihuan/Dev/github/adk_tutor/.venv/lib/python3.13/site-packages/google/adk/sessions/__init__.py)


✅ 從子模組 import 成功: SqliteSessionService


In [11]:
import tempfile
from pathlib import Path

db_path = Path(tempfile.gettempdir()) / "adk_tutor_demo.db"
db_path.unlink(missing_ok=True)

# 注意：SqliteSessionService 收的是 db_path（純路徑字串），
# 不是 DatabaseSessionService 那種 "sqlite:///..." 的 URL。傳錯會 TypeError。
persistent = SqliteSessionService(db_path=str(db_path))
disk_runner = Runner(agent=assistant, app_name=APP, session_service=persistent)

sid_disk = await new_session(disk_runner)
await ask(disk_runner, "請記住：我的專案代號是 Falcon。", session_id=sid_disk)

# 模擬「重開機」：丟掉舊物件，用同一個檔案重新連
reopened = SqliteSessionService(db_path=str(db_path))
runner2 = Runner(agent=assistant, app_name=APP, session_service=reopened)

print("重新連線後提問:", await ask(runner2, "我的專案代號是什麼？", session_id=sid_disk))
print(f"\n資料庫檔案: {db_path}（{db_path.stat().st_size} bytes）")

重新連線後提問: 你的專案代號是 Falcon。

資料庫檔案: /tmp/adk_tutor_demo.db（36864 bytes）


對話活過了「重開機」。而 `assistant` 這個 agent 從頭到尾一行都沒改——
**持久化是 Runner 的事，不是 agent 的事**。

## 本章重點

- **Session = 一段對話**，身上有 `events`（發生過什麼）和 `state`（現在怎樣）。
- **agent 的記憶就是 events 被整包重送**。長對話越來越貴的原因就在這。
- **State 前綴決定作用域**：無前綴＝本 session、`user:`＝跨 session、
  `app:`＝跨使用者、`temp:`＝**不會被存下來**。
- **`output_key`** 把 agent 回覆自動寫進 state，是 pipeline 傳資料的標準做法。
- **不要直接改 `session.state`**，要用 `tool_context.state` 或 `EventActions(state_delta=...)`。
- **換持久化只要換 SessionService**，agent 程式碼不動。
  注意 `SqliteSessionService` 得從子模組 import，而且參數是 `db_path` 不是 `db_url`。

## 動手練習

1. 把第 2 節的 `user:` 前綴拿掉，重跑，確認 session B 就讀不到了。
2. 用 `temp:` 存一個值，然後在同一次呼叫的另一個工具裡把它讀出來——
   確認它在「同一次呼叫內」是有效的。
3. 第 5 節換成 `DatabaseSessionService(db_url=f"sqlite:///{db_path}")`
   ——注意參數名不一樣——確認兩種實作可以讀到彼此的資料。

---
**下一站 → `05_memory_artifacts.ipynb`**：跨對話的長期記憶，以及檔案怎麼存。